In [3]:
# ============================================================
# CREDRESOLVE — COLLECTIONS RECOVERY ANALYTICS
# 04 — ASSIGNMENT FORENSIC ANALYSIS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 90)
print("CREDRESOLVE — ASSIGNMENT FORENSIC ANALYSIS")
print("=" * 90)


# ------------------------------------------------------------
# 2. LOAD RAW DATA
# ------------------------------------------------------------

csv_files = sorted(RAW_DIR.glob("*.csv"))

datasets = {
    file.stem: pd.read_csv(file)
    for file in csv_files
}

print(f"Datasets loaded: {len(datasets)}")


# ============================================================
# A. DUPLICATE PAYMENT FORENSICS
# ============================================================

print("\n" + "=" * 90)
print("A. DUPLICATE PAYMENT FORENSICS")
print("=" * 90)

payments = datasets["payments"].copy()

payments["event_at_parsed"] = pd.to_datetime(
    payments["event_at"],
    errors="coerce"
)

# A1. Duplicate payment IDs
payment_id_profile = (
    payments
    .groupby("payment_id")
    .agg(
        occurrences=("payment_id", "size"),
        accounts=("account_id", "nunique"),
        borrowers=("borrower_id", "nunique"),
        references=("payment_reference", "nunique"),
        amounts=("amount", "nunique"),
        providers=("provider_id", "nunique")
    )
    .reset_index()
)

duplicate_payment_ids = payment_id_profile[
    payment_id_profile["occurrences"] > 1
].copy()

print("Duplicate payment IDs:", len(duplicate_payment_ids))

display(duplicate_payment_ids.head(50))

duplicate_payment_ids.to_csv(
    OUTPUT_DIR / "forensic_payment_id_duplicates.csv",
    index=False
)


# A2. Duplicate payment references
reference_profile = (
    payments
    .dropna(subset=["payment_reference"])
    .groupby("payment_reference")
    .agg(
        occurrences=("payment_reference", "size"),
        payment_ids=("payment_id", "nunique"),
        accounts=("account_id", "nunique"),
        amounts=("amount", "nunique"),
        providers=("provider_id", "nunique")
    )
    .reset_index()
)

duplicate_references = reference_profile[
    reference_profile["occurrences"] > 1
].copy()

print(
    "Duplicate payment references:",
    len(duplicate_references)
)

display(duplicate_references.head(50))

duplicate_references.to_csv(
    OUTPUT_DIR / "forensic_payment_reference_duplicates.csv",
    index=False
)


# A3. Same account + amount + timestamp
payment_event_duplicates = (
    payments
    .dropna(
        subset=[
            "account_id",
            "amount",
            "event_at_parsed"
        ]
    )
    .groupby(
        [
            "account_id",
            "amount",
            "event_at_parsed"
        ]
    )
    .size()
    .reset_index(name="occurrences")
)

payment_event_duplicates = (
    payment_event_duplicates[
        payment_event_duplicates["occurrences"] > 1
    ]
    .sort_values("occurrences", ascending=False)
)

print(
    "Same account + amount + timestamp groups:",
    len(payment_event_duplicates)
)

display(payment_event_duplicates.head(50))

payment_event_duplicates.to_csv(
    OUTPUT_DIR / "forensic_payment_event_duplicates.csv",
    index=False
)


# ============================================================
# B. ATTRIBUTION FORENSICS
# ============================================================

print("\n" + "=" * 90)
print("B. ATTRIBUTION FORENSICS")
print("=" * 90)

calls = datasets["calls"].copy()

calls["event_at_parsed"] = pd.to_datetime(
    calls["event_at"],
    errors="coerce"
)

accounts = datasets["accounts"].copy()


# B1. Payment borrower/account consistency

account_borrower_map = (
    accounts[
        [
            "account_id",
            "borrower_id"
        ]
    ]
    .drop_duplicates("account_id")
    .rename(
        columns={
            "borrower_id": "account_borrower_id"
        }
    )
)

payment_account_check = payments.merge(
    account_borrower_map,
    on="account_id",
    how="left"
)

payment_account_check[
    "borrower_mismatch"
] = (
    payment_account_check["borrower_id"]
    !=
    payment_account_check["account_borrower_id"]
)

borrower_mismatch = payment_account_check[
    payment_account_check["borrower_mismatch"]
    &
    payment_account_check["account_borrower_id"].notna()
].copy()

print(
    "Payment borrower/account mismatches:",
    len(borrower_mismatch)
)

display(borrower_mismatch.head(50))

borrower_mismatch.to_csv(
    OUTPUT_DIR / "forensic_payment_borrower_mismatch.csv",
    index=False
)


# B2. Call borrower/account consistency

call_account_check = calls.merge(
    account_borrower_map,
    on="account_id",
    how="left"
)

call_account_check[
    "borrower_mismatch"
] = (
    call_account_check["borrower_id"]
    !=
    call_account_check["account_borrower_id"]
)

call_borrower_mismatch = call_account_check[
    call_account_check["borrower_mismatch"]
    &
    call_account_check["account_borrower_id"].notna()
].copy()

print(
    "Call borrower/account mismatches:",
    len(call_borrower_mismatch)
)

display(call_borrower_mismatch.head(50))

call_borrower_mismatch.to_csv(
    OUTPUT_DIR / "forensic_call_borrower_mismatch.csv",
    index=False
)


# B3. Payment → prior call attribution signal
# This is an attribution SIGNAL, not proof of causality.

payments_match = payments[
    [
        "payment_id",
        "account_id",
        "event_at_parsed",
        "amount",
        "payment_status"
    ]
].dropna(
    subset=[
        "account_id",
        "event_at_parsed"
    ]
).sort_values("event_at_parsed")

calls_match = calls[
    [
        "call_id",
        "account_id",
        "event_at_parsed",
        "agent_id",
        "campaign_id",
        "vendor_id",
        "call_status"
    ]
].dropna(
    subset=[
        "account_id",
        "event_at_parsed"
    ]
).rename(
    columns={
        "event_at_parsed": "call_event_at"
    }
).sort_values("call_event_at")

attribution_signal = pd.merge_asof(
    payments_match.sort_values("event_at_parsed"),
    calls_match.sort_values("call_event_at"),
    left_on="event_at_parsed",
    right_on="call_event_at",
    by="account_id",
    direction="backward"
)

attribution_signal["hours_since_prior_call"] = (
    attribution_signal["event_at_parsed"]
    -
    attribution_signal["call_event_at"]
).dt.total_seconds() / 3600

print(
    "Payment-to-prior-call attribution records:",
    len(attribution_signal)
)

display(attribution_signal.head(50))

attribution_signal.to_csv(
    OUTPUT_DIR / "forensic_payment_call_attribution.csv",
    index=False
)


# ============================================================
# C. TIMEZONE FORENSICS
# ============================================================

print("\n" + "=" * 90)
print("C. TIMEZONE FORENSICS")
print("=" * 90)

# C1. Profile all actual timezone fields

timezone_profiles = []

for table, column in [
    ("calls", "timezone"),
    ("accounts", "timezone"),
    ("agent_sessions", "timezone"),
    ("vendor_telephony", "timezone")
]:

    df = datasets[table]

    profile = (
        df[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="records")
    )

    profile["dataset"] = table

    timezone_profiles.append(profile)

timezone_profile_df = pd.concat(
    timezone_profiles,
    ignore_index=True
)

display(timezone_profile_df)

timezone_profile_df.to_csv(
    OUTPUT_DIR / "forensic_timezone_profile.csv",
    index=False
)


# C2. Correct comparison:
# Call timezone vs ACCOUNT timezone.
#
# Vendor timezone is NOT treated as the expected customer/call
# timezone because vendor timezone represents the vendor system.

account_timezone_map = (
    accounts[
        [
            "account_id",
            "timezone"
        ]
    ]
    .drop_duplicates("account_id")
    .rename(
        columns={
            "timezone": "account_timezone"
        }
    )
)

call_timezone_check = calls.merge(
    account_timezone_map,
    on="account_id",
    how="left"
)

call_timezone_check[
    "account_timezone_difference_flag"
] = (
    call_timezone_check["timezone"]
    !=
    call_timezone_check["account_timezone"]
)

call_timezone_check[
    "account_timezone_difference_flag"
] = (
    call_timezone_check[
        "account_timezone_difference_flag"
    ]
    &
    call_timezone_check["timezone"].notna()
    &
    call_timezone_check["account_timezone"].notna()
)

timezone_differences = call_timezone_check[
    call_timezone_check[
        "account_timezone_difference_flag"
    ]
].copy()

print(
    "Call/account timezone differences:",
    len(timezone_differences)
)

display(timezone_differences.head(50))

timezone_differences.to_csv(
    OUTPUT_DIR / "forensic_call_account_timezone_difference.csv",
    index=False
)


# C3. Calling-hour distribution

calling_hour_profile = (
    calls["event_at_parsed"]
    .dt.hour
    .value_counts()
    .sort_index()
    .rename_axis("hour")
    .reset_index(name="call_count")
)

display(calling_hour_profile)

calling_hour_profile.to_csv(
    OUTPUT_DIR / "forensic_calling_hours.csv",
    index=False
)


# ============================================================
# D. VENDOR MAPPING / VERSION FORENSICS
# ============================================================

print("\n" + "=" * 90)
print("D. VENDOR MAPPING / VERSION FORENSICS")
print("=" * 90)

vendor = datasets["vendor_telephony"].copy()

vendor_profile = (
    vendor[
        [
            "vendor_id",
            "vendor_name",
            "vendor_account_id",
            "timezone",
            "status",
            "schema_version"
        ]
    ]
    .drop_duplicates()
)

display(vendor_profile)

vendor_profile.to_csv(
    OUTPUT_DIR / "forensic_vendor_master_profile.csv",
    index=False
)


# D2. Calls by vendor and schema version

calls_vendor = calls.merge(
    vendor[
        [
            "vendor_id",
            "schema_version"
        ]
    ],
    on="vendor_id",
    how="left"
)

calls_vendor["month"] = (
    calls_vendor["event_at_parsed"]
    .dt.to_period("M")
    .astype(str)
)

vendor_monthly_profile = (
    calls_vendor
    .groupby(
        [
            "month",
            "vendor_id",
            "schema_version"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="call_count")
)

display(vendor_monthly_profile)

vendor_monthly_profile.to_csv(
    OUTPUT_DIR / "forensic_vendor_monthly_profile.csv",
    index=False
)


# D3. Disposition version profile

dispositions = datasets["call_dispositions"].copy()

dispositions["event_at_parsed"] = pd.to_datetime(
    dispositions["event_at"],
    errors="coerce"
)

dispositions["month"] = (
    dispositions["event_at_parsed"]
    .dt.to_period("M")
    .astype(str)
)

disposition_version_profile = (
    dispositions
    .groupby(
        [
            "month",
            "disposition_version",
            "disposition_code"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="records")
)

display(disposition_version_profile)

disposition_version_profile.to_csv(
    OUTPUT_DIR / "forensic_disposition_version_profile.csv",
    index=False
)


# ============================================================
# E. AGENT IDENTITY FORENSICS
# ============================================================

print("\n" + "=" * 90)
print("E. AGENT IDENTITY FORENSICS")
print("=" * 90)

agents = datasets["agents"].copy()
sessions = datasets["agent_sessions"].copy()


# E1. Agent identity consistency

agent_identity_profile = (
    agents
    .groupby("agent_id")
    .agg(
        employee_codes=("employee_code", "nunique"),
        names=("agent_name", "nunique"),
        vendors=("vendor_id", "nunique"),
        teams=("team", "nunique"),
        statuses=("status", "nunique")
    )
    .reset_index()
)

agent_identity_flags = agent_identity_profile[
    (
        (agent_identity_profile["employee_codes"] > 1)
        |
        (agent_identity_profile["names"] > 1)
        |
        (agent_identity_profile["vendors"] > 1)
        |
        (agent_identity_profile["teams"] > 1)
    )
].copy()

print(
    "Agent identity flags:",
    len(agent_identity_flags)
)

display(agent_identity_flags)

agent_identity_flags.to_csv(
    OUTPUT_DIR / "forensic_agent_identity_flags.csv",
    index=False
)


# E2. Agent session overlaps

sessions["login_at"] = pd.to_datetime(
    sessions["login_at"],
    errors="coerce"
)

sessions["logout_at"] = pd.to_datetime(
    sessions["logout_at"],
    errors="coerce"
)

session_overlap_flags = []

for agent_id, group in sessions.dropna(
    subset=[
        "agent_id",
        "login_at",
        "logout_at"
    ]
).groupby("agent_id"):

    group = group.sort_values("login_at")

    previous_logout = None
    previous_session = None

    for _, row in group.iterrows():

        if (
            previous_logout is not None
            and row["login_at"] < previous_logout
        ):

            session_overlap_flags.append({
                "agent_id": agent_id,
                "previous_session_id": previous_session,
                "current_session_id": row["session_id"],
                "previous_logout": previous_logout,
                "current_login": row["login_at"]
            })

        if (
            previous_logout is None
            or row["logout_at"] > previous_logout
        ):
            previous_logout = row["logout_at"]
            previous_session = row["session_id"]

session_overlap_df = pd.DataFrame(
    session_overlap_flags
)

print(
    "Agent session overlaps:",
    len(session_overlap_df)
)

display(session_overlap_df.head(50))

session_overlap_df.to_csv(
    OUTPUT_DIR / "forensic_agent_session_overlaps.csv",
    index=False
)


# E3. Calls with unknown agent

agent_ids = set(
    agents["agent_id"].dropna()
)

unknown_call_agents = calls[
    calls["agent_id"].notna()
    &
    ~calls["agent_id"].isin(agent_ids)
].copy()

print(
    "Calls with unknown agent IDs:",
    len(unknown_call_agents)
)

display(unknown_call_agents.head(50))

unknown_call_agents.to_csv(
    OUTPUT_DIR / "forensic_unknown_call_agents.csv",
    index=False
)


# ============================================================
# F. PORTFOLIO MIX FORENSICS
# ============================================================

print("\n" + "=" * 90)
print("F. PORTFOLIO MIX FORENSICS")
print("=" * 90)

accounts["opened_at_parsed"] = pd.to_datetime(
    accounts["opened_at"],
    errors="coerce"
)

accounts["month"] = (
    accounts["opened_at_parsed"]
    .dt.to_period("M")
    .astype(str)
)

portfolio_dimensions = [
    "loan_type",
    "dpd",
    "risk_segment",
    "status",
    "schema_version"
]

portfolio_profiles = []

for column in portfolio_dimensions:

    profile = (
        accounts
        .groupby(
            [
                "month",
                column
            ],
            dropna=False
        )
        .size()
        .reset_index(name="account_count")
    )

    profile["share_of_month"] = (
        profile
        .groupby("month")["account_count"]
        .transform(
            lambda x: x / x.sum()
        )
    )

    profile["dimension"] = column

    portfolio_profiles.append(profile)

portfolio_mix_df = pd.concat(
    portfolio_profiles,
    ignore_index=True
)

display(portfolio_mix_df)

portfolio_mix_df.to_csv(
    OUTPUT_DIR / "forensic_portfolio_mix.csv",
    index=False
)


# ============================================================
# G. DENOMINATOR / TARGETING FORENSICS
# ============================================================

print("\n" + "=" * 90)
print("G. DENOMINATOR / TARGETING FORENSICS")
print("=" * 90)

targeting = datasets["daily_targeting"].copy()

targeting["target_date_parsed"] = pd.to_datetime(
    targeting["target_date"],
    errors="coerce"
)

targeting["month"] = (
    targeting["target_date_parsed"]
    .dt.to_period("M")
    .astype(str)
)


# G1. Target population

target_population = (
    targeting
    .groupby("month")
    .agg(
        targeting_rows=("target_id", "size"),
        unique_targets=("target_id", "nunique"),
        unique_accounts=("account_id", "nunique"),
        unique_campaigns=("campaign_id", "nunique")
    )
    .reset_index()
)

target_population[
    "account_population_change_pct"
] = (
    target_population["unique_accounts"]
    .pct_change() * 100
)

display(target_population)

target_population.to_csv(
    OUTPUT_DIR / "forensic_target_population.csv",
    index=False
)


# G2. Target status

target_status_profile = (
    targeting
    .groupby(
        [
            "month",
            "status"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="target_count")
)

target_status_profile[
    "share_within_month"
] = (
    target_status_profile
    .groupby("month")["target_count"]
    .transform(
        lambda x: x / x.sum()
    )
)

display(target_status_profile)

target_status_profile.to_csv(
    OUTPUT_DIR / "forensic_target_status_profile.csv",
    index=False
)


# G3. Target priority

target_priority_profile = (
    targeting
    .groupby(
        [
            "month",
            "priority"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="target_count")
)

target_priority_profile[
    "share_within_month"
] = (
    target_priority_profile
    .groupby("month")["target_count"]
    .transform(
        lambda x: x / x.sum()
    )
)

display(target_priority_profile)

target_priority_profile.to_csv(
    OUTPUT_DIR / "forensic_target_priority_profile.csv",
    index=False
)


# G4. Account population vs targeted population

account_monthly_population = (
    accounts
    .groupby("month")
    .agg(
        account_population=(
            "account_id",
            "nunique"
        )
    )
    .reset_index()
)

denominator_comparison = (
    account_monthly_population
    .merge(
        target_population[
            [
                "month",
                "unique_accounts",
                "unique_campaigns"
            ]
        ],
        on="month",
        how="outer"
    )
    .sort_values("month")
)

denominator_comparison[
    "targeted_share_of_accounts"
] = (
    denominator_comparison["unique_accounts"]
    /
    denominator_comparison["account_population"]
)

display(denominator_comparison)

denominator_comparison.to_csv(
    OUTPUT_DIR / "forensic_denominator_comparison.csv",
    index=False
)


# ============================================================
# H. FORENSIC SUMMARY
# ============================================================

print("\n" + "=" * 90)
print("FORENSIC INVESTIGATION SUMMARY")
print("=" * 90)

summary = pd.DataFrame([{

    "duplicate_payment_ids":
        len(duplicate_payment_ids),

    "duplicate_payment_references":
        len(duplicate_references),

    "duplicate_payment_event_groups":
        len(payment_event_duplicates),

    "payment_borrower_mismatches":
        len(borrower_mismatch),

    "call_borrower_mismatches":
        len(call_borrower_mismatch),

    "call_account_timezone_differences":
        len(timezone_differences),

    "agent_identity_flags":
        len(agent_identity_flags),

    "agent_session_overlaps":
        len(session_overlap_df),

    "unknown_call_agents":
        len(unknown_call_agents),

    "portfolio_dimensions_reviewed":
        len(portfolio_dimensions),

    "months_in_target_population":
        len(target_population),

    "months_in_denominator_comparison":
        len(denominator_comparison)
}])

display(summary)

summary.to_csv(
    OUTPUT_DIR / "forensic_summary_corrected.csv",
    index=False
)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 90)
print("ASSIGNMENT FORENSICS COMPLETE")
print("=" * 90)

print(
    "Seven assignment-required forensic areas were investigated."
)

print(
    "Raw source files were NOT modified."
)

print(
    f"Forensic outputs saved to: {OUTPUT_DIR}"
)

print(
    "Findings are evidence for investigation, "
    "not automatic proof of fraud, error, causality, "
    "or manipulation."
)

CREDRESOLVE — ASSIGNMENT FORENSIC ANALYSIS
Datasets loaded: 18

A. DUPLICATE PAYMENT FORENSICS
Duplicate payment IDs: 500


,payment_id,occurrences,accounts,borrowers,references,amounts,providers
55,PAYMENT0000056,2,1,1,1,1,1
75,PAYMENT0000076,2,1,1,1,1,1
107,PAYMENT0000108,2,1,1,1,1,1
148,PAYMENT0000149,2,1,1,1,1,1
198,PAYMENT0000199,2,1,1,1,1,1
254,PAYMENT0000255,2,1,1,1,1,1
262,PAYMENT0000263,2,1,1,1,1,1
310,PAYMENT0000311,2,1,1,1,1,1
551,PAYMENT0000552,2,1,1,1,1,1
552,PAYMENT0000553,2,1,1,1,1,1


Duplicate payment references: 3745


,payment_reference,occurrences,payment_ids,accounts,amounts,providers
0,TXN0000000009,2,1,1,1,1
5,TXN0000000027,2,2,2,2,2
8,TXN0000000032,3,3,3,3,3
28,TXN0000000113,3,3,3,3,3
35,TXN0000000132,2,2,2,2,2
36,TXN0000000134,2,2,2,2,2
39,TXN0000000154,2,2,2,2,2
50,TXN0000000186,2,2,2,2,2
53,TXN0000000196,2,2,2,2,2
58,TXN0000000212,2,2,2,2,2


Same account + amount + timestamp groups: 500


...

,month,priority,target_count,share_within_month
0,2026-01,1,614,0.096404
1,2026-01,2,644,0.101115
2,2026-01,3,636,0.099859
3,2026-01,4,639,0.100330
4,2026-01,5,608,0.095462
...,...,...,...,...
75,2026-08,6,141,0.088070
76,2026-08,7,151,0.094316
77,2026-08,8,188,0.117427
78,2026-08,9,163,0.101811


,month,account_population,unique_accounts,unique_campaigns,targeted_share_of_accounts
0,2024-01,1311.0,NaN,NaN,NaN
1,2024-02,1299.0,NaN,NaN,NaN
2,2024-03,1404.0,NaN,NaN,NaN
3,2024-04,1263.0,NaN,NaN,NaN
4,2024-05,1287.0,NaN,NaN,NaN
5,2024-06,1219.0,NaN,NaN,NaN
6,2024-07,1324.0,NaN,NaN,NaN
7,2024-08,1358.0,NaN,NaN,NaN
8,2024-09,1326.0,NaN,NaN,NaN
9,2024-10,1362.0,NaN,NaN,NaN



FORENSIC INVESTIGATION SUMMARY


,duplicate_payment_ids,duplicate_payment_references,duplicate_payment_event_groups,payment_borrower_mismatches,call_borrower_mismatches,call_account_timezone_differences,agent_identity_flags,agent_session_overlaps,unknown_call_agents,portfolio_dimensions_reviewed,months_in_target_population,months_in_denominator_comparison
0,500,3745,500,25113,89939,60887,1000,215,0,5,8,31



ASSIGNMENT FORENSICS COMPLETE
Seven assignment-required forensic areas were investigated.
Raw source files were NOT modified.
Forensic outputs saved to: c:\Users\DELL\Documents\CredResolve Collections Recovery Analytics\outputs\tables
Findings are evidence for investigation, not automatic proof of fraud, error, causality, or manipulation.
